In [1]:
from model.model import get_model
import os
from dotenv import load_dotenv
from outlines import Generator, from_transformers  
from schema.ticket import Ticket 
from prompt.summarizer import summary_prompt
from schema.table import Table
import json
from evalution.metric import TicketEvaluator
load_dotenv()

True

In [2]:
location = os.getenv('DATA_FILE_NAME')
table_1 = Table(location)

In [3]:
name = os.getenv('MODEL_NAME')
hf_model,hf_tokenizer = get_model(name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model downloaded


In [4]:
hf_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [5]:
model = from_transformers(hf_model,hf_tokenizer)

In [6]:
generator = Generator(model,Ticket)

In [7]:
evaluator = TicketEvaluator(generator=generator)


In [8]:
#metric = evaluator.evalute_ticket(table_1.return_ticket(1),max_new_tokens=150,use_cache= True)

In [9]:
df = evaluator.evalutaion_dataframe(max_new_tokens=150,use_cache= True)

ticket_id                                                    1
message      My laptop screen is broken and I need urgent h...
category                                             technical
sentiment                                             negative
urgency                                                   high
Name: 0, dtype: object


W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] WON'T CONVERT _apply_token_bitmask_inplace_kernel d:\Programing\Depi\ticket_summary\ticket_summary\.venv\Lib\site-packages\outlines_core\kernels\torch.py line 43 
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] due to: 
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] Traceback (most recent call last):
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415]   File "d:\Programing\Depi\ticket_summary\ticket_summary\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2319, in __call__
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415]     result = self._inner_convert(
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\convert_frame.py:2415]         frame, cache_entry, hooks, frame_state, skip=skip + 1
W0910 16:51:38.764000 33248 Lib\site-packages\torch\_dynamo\co

ticket_id                                             2
message      I forgot my password and need to reset it.
category                                        account
sentiment                                       neutral
urgency                                          medium
Name: 1, dtype: object
ticket_id                                    3
message      My package arrived two days late.
category                              delivery
sentiment                             negative
urgency                                 medium
Name: 2, dtype: object
ticket_id                                          4
message      I was charged twice for the same order.
category                                     billing
sentiment                                   negative
urgency                                         high
Name: 3, dtype: object
ticket_id                                    5
message      I want to cancel my subscription.
category                          subscription
sentiment 

In [18]:
df

,ticket_id,message,category,sentiment,urgency,success,pred_category,pred_sentiment,pred_urgency,pred_summary,category_accuracy,sentiment_accuracy,urgency_accuracy
0,1,My laptop screen is broken and I need urgent h...,technical,negative,high,True,technical,negative,high,Customer needs urgent technical support due to...,True,True,True
1,2,I forgot my password and need to reset it.,account,neutral,medium,True,account,neutral,low,Customer wants assistance with resetting their...,True,True,False
2,3,My package arrived two days late.,delivery,negative,medium,True,delivery,negative,high,Package delay after expected delivery time.,True,True,False
3,4,I was charged twice for the same order.,billing,negative,high,True,billing,negative,high,Customer had to pay twice for the same order.,True,True,True
4,5,I want to cancel my subscription.,subscription,neutral,medium,True,subscription,negative,low,Customer wants to end their current subscripti...,True,False,False
5,6,"Your support team was very helpful, thank you!",account,positive,low,True,subscription,positive,low,Customer appreciated the support from their su...,False,True,True
6,7,The payment failed but money was deducted from...,billing,negative,high,True,technical,neutral,high,Payment failed and funds were debited from you...,False,False,True
7,8,I received my order early and everything looks...,delivery,positive,low,True,delivery,positive,low,Order delivered on time with satisfactory qual...,True,True,True
8,9,The mobile app freezes when I upload a file.,technical,negative,high,True,technical,negative,high,File upload causing freeze on mobile app,True,True,True
9,10,How can I upgrade my subscription plan?,subscription,neutral,low,True,technical,neutral,high,Customer needs help upgrading their subscripti...,False,True,False


In [15]:
metric=evaluator.accuracy_metric(df)

clean_metric = {k:float(v) for k,v in metric.items()}

In [16]:
clean_metric

{'category_accuracy': 70.0,
 'sentiment_accuracy': 80.0,
 'urgency_accuracy': 60.0}

In [17]:
#result = generator(summary_prompt(table_1.return_ticket(2)), max_new_tokens=150,use_cache= True)
#print(result)

In [13]:
output = Ticket.model_validate_json(result)
output = output.model_dump_json()

NameError: name 'result' is not defined

In [ ]:
output = json.loads(output)

In [ ]:
output['category']

'delivery'

In [ ]:
table_1.return_row(1)

,ticket_id,message,category,sentiment,urgency
8,9,The mobile app freezes when I upload a file.,technical,negative,high
